In [ ]:
from google import genai
import sys, os
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)
import json
import os
from PIL import Image
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText
from utils.img_utils import resize_long_edge
from utils.caption_utils import scene_generate_v2, reason_generate

from nuscenes.nuscenes import NuScenes
from data.nuscenes_data import NuscenesData
from pathlib import Path

import numpy as np
from IPython.display import display, Markdown
import matplotlib.pyplot as plt
import re

In [ ]:
# Load dataset
data_path = Path("/home/ximeng/Dataset/nuscenes_full_v1_0/")
nusc = NuScenes(version='v1.0-mini', dataroot=data_path)
is_train = 0 # 0: train, 1: val, 2: test
pre_frame = 4 # 2s
future_frame = 12 # 6s
dataset = NuscenesData(nusc, is_train, pre_frame, future_frame)

target_index = 10 # 155
sample = dataset[target_index]
token = sample['token']
# Images and LiDAR
raw_images = sample['raw_images']
raw_lidar = sample['raw_lidar']
# Ego state
pre_waypoints = sample['pre_waypoints']
velocity = sample['velocity']
acceleration = sample['accel']
yaw_rate = sample['yaw_rate']
# High-level command
command = sample['command']

instance = sample['instance']
# Ground truth
future_waypoints = sample['future_waypoints']

In [14]:
client = genai.Client(
    vertexai=True,
    project="norse-limiter-487520-c0",
    location="us-south1" # Dallas
)

In [15]:
try:
    # Use gemini-1.5-flash which is confirmed to be stable
    response = client.models.generate_content(
        model="gemini-2.5-flash", 
        contents="Hello, world! This is a test.",
    )
    print(response.text)
except Exception as e:
    print(f"Error: {e}")

/home/ximeng/anaconda3/envs/dllm/lib/python3.10/site-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Hello! I received your test message loud and clear. How can I help you today?
